['52. 2016_DDR-2016-Rubis-UK.json'] error with this file

In [2]:
from utils import get_tables
import json
import os
import pandas as pd

In [ ]:
INCLUDE_TABLES = False # save space and stored in /data/ReportJson
EXPORT_TO_XLSX = True # to make it easier to export, agg in later steps
json_reports_path = "./data/ReportJson"

for path in os.listdir(json_reports_path):
    PDF = os.path.basename(PDF)
    REPORT_JSON_PATH = f"./data/ReportJson/{os.path.splitext(PDF)[0]}.json"
    with open(REPORT_JSON_PATH, 'r') as f: 
        document = json.load(f)

    OUT_JSON_PATH = f"./out/json/{os.path.splitext(PDF)[0]}_sources.json"
    out_data = {}
    if os.path.exists(OUT_JSON_PATH): 
        with open(OUT_JSON_PATH, 'r') as f:
            out_data = json.load(f)

    table_data = get_tables.main(document)
    out_data['tables'] = table_data

    table_refs = set(table_data['all table refs'])
    relevant_tables = [t for t in document['tables'] if t['self_ref'] in table_refs]

    if INCLUDE_TABLES: 
        out_data['tables']['tables'] = relevant_tables

    with open(OUT_JSON_PATH, 'w') as f: 
        json.dump(out_data, f)

    if EXPORT_TO_XLSX:
        with pd.ExcelWriter(f"./out/xlsx/{os.path.splitext(PDF)[0]}_tables.xlsx") as writer: 
            for i, t in enumerate(relevant_tables):
                df = get_tables.extract_table_to_df(t) 
                df.to_excel(writer, sheet_name=f"Sheet_{i}", index=False)

In [ ]:
d = {}
for p in os.listdir('./out/json/'):  
    with open(f"./out/json/{p}", 'r') as f:
        out_data = json.load(f)
    missing = out_data['tables']["keywords missing tables"]
    l = len(missing)
    if l not in d: 
        d[l] = 0
    d[l] += 1

{2: 32, 0: 93, 4: 18, 3: 26, 5: 7, 1: 58, 6: 7, 9: 7, 8: 1}


In [ ]:
# 
[print(f"{v} missing {k} tables") for k,v in d.items()]

32 missing 2 tables
93 missing 0 tables
18 missing 4 tables
26 missing 3 tables
7 missing 5 tables
58 missing 1 tables
7 missing 6 tables
7 missing 9 tables
1 missing 8 tables


[None, None, None, None, None, None, None, None, None]

In [10]:
tables = {}
for p in os.listdir('./out/json/'):  
    with open(f"./out/json/{p}", 'r') as f:
        out_data = json.load(f)
    missing = out_data['tables']["keywords missing tables"]
    for v in missing: 
        if v not in tables: 
            tables[v] = 0
        tables[v] += 1
tables

{'Biodiversity': 109,
 'Environmental Grievance Mechanism': 97,
 'Raw Materials and Packing': 45,
 'Environmental Compliance': 50,
 'Waste and Spills': 37,
 'GHG Emissions': 11,
 'Water': 22,
 'Other Air Emissions': 37,
 'Energy': 12}

In [14]:
per_rep = {}
for p in os.listdir('./out/json/'):  
    with open(f"./out/json/{p}", 'r') as f:
        out_data = json.load(f)
    rep_num = p[:2]
    if rep_num not in per_rep: 
        per_rep[rep_num] = dict()
        per_rep[rep_num]['count'] = 0
    per_rep[rep_num]['count'] += 1

    missing = out_data['tables']["keywords missing tables"]
    for v in missing: 
        if v not in per_rep[rep_num]: 
            per_rep[rep_num][v] = 0
        per_rep[rep_num][v] += 1

errors_per_report = {rep_num:{k:v/per_rep[rep_num]['count'] for k,v in per_rep[rep_num].items()} for rep_num in per_rep.keys()}
# what percent of reports per company weren't able to extract a table for a given category
errors_per_report

{'59': {'count': 1.0,
  'Biodiversity': 0.42857142857142855,
  'Environmental Grievance Mechanism': 0.5714285714285714,
  'Environmental Compliance': 0.2857142857142857,
  'Water': 0.14285714285714285},
 '56': {'count': 1.0,
  'Raw Materials and Packing': 0.3333333333333333,
  'Environmental Grievance Mechanism': 0.3333333333333333},
 '38': {'count': 1.0},
 '35': {'count': 1.0,
  'Environmental Compliance': 0.2,
  'Environmental Grievance Mechanism': 0.8,
  'Raw Materials and Packing': 0.4,
  'Biodiversity': 0.2,
  'Other Air Emissions': 0.1},
 '42': {'count': 1.0,
  'Biodiversity': 1.0,
  'Waste and Spills': 0.5,
  'Environmental Compliance': 0.4,
  'Environmental Grievance Mechanism': 0.9,
  'Water': 0.4},
 '51': {'count': 1.0,
  'GHG Emissions': 0.14285714285714285,
  'Water': 0.2857142857142857,
  'Biodiversity': 0.2857142857142857,
  'Environmental Compliance': 0.5714285714285714,
  'Environmental Grievance Mechanism': 0.7142857142857143,
  'Raw Materials and Packing': 0.285714285

In [ ]:
# 1 means complete success, 
[(i, sum(1 for e in errors_per_report.values() if len(e) <=i ) / (33)) for i in range(0,11)]

[(0, 0.0),
 (1, 0.09090909090909091),
 (2, 0.21212121212121213),
 (3, 0.30303030303030304),
 (4, 0.42424242424242425),
 (5, 0.5454545454545454),
 (6, 0.6666666666666666),
 (7, 0.7575757575757576),
 (8, 0.8181818181818182),
 (9, 0.8484848484848485),
 (10, 1.0)]

In [2]:
v = [(0, 0.0),
 (1, 0.09090909090909091),
 (2, 0.21212121212121213),
 (3, 0.30303030303030304),
 (4, 0.42424242424242425),
 (5, 0.5454545454545454),
 (6, 0.6666666666666666),
 (7, 0.7575757575757576),
 (8, 0.8181818181818182),
 (9, 0.8484848484848485),
 (10, 1.0)]
for i in range(1, 11): 
    print(f"missing {10-i}", v[i][1] - v[i-1][1])


missing 9 0.09090909090909091
missing 8 0.12121212121212122
missing 7 0.09090909090909091
missing 6 0.12121212121212122
missing 5 0.12121212121212116
missing 4 0.12121212121212122
missing 3 0.09090909090909094
missing 2 0.06060606060606066
missing 1 0.030303030303030276
missing 0 0.1515151515151515
